<a href="https://colab.research.google.com/github/TCU-DCDA/WRIT20833_2026/blob/main/notebooks/homework/WRIT20833_HW4_2026_ANSWER_KEY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 4 — Topic Modeling & Integration — INSTRUCTOR ANSWER KEY

**WRIT 20833 — Intro to Coding in the Humanities**

> Solutions are one valid approach. **Ungrading:** this key is a **worked example + discussion
> fodder, not a rubric** — reward how students *name and doubt* topics, not "correct" ones.
>
> ⚠️ **LDA is stochastic + version-sensitive.** Unlike HW2/HW3 (exact, reproducible numbers), the
> topic words, per-topic counts, and per-topic sentiment **will differ** by machine, gensim version,
> and run — even with `random_state=42`. So this key states exact values only for the **deterministic**
> steps (vocabulary size, the `[]` edge case) and describes LDA output **qualitatively**; the value is
> in the *method and interpretation*. The full pipeline was validated to run top-to-bottom on the
> course corpus (gensim 4.4.0; **123 rows**).

---

In [ ]:
# SETUP — run this cell first. (You're not expected to read every line yet — that's fine.)
import re, os, urllib.request
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt

# Two borrowed tools: VADER (sentiment, from HW3) and Gensim (topic modeling, new this week).
!pip install -q gensim vaderSentiment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from gensim import corpora
from gensim.models import LdaModel
analyzer = SentimentIntensityAnalyzer()

# Same word-splitter + stopwords from HW2/HW3 (we reuse our own tools).
def split_into_words(any_chunk_of_text):
    return re.split(r"\W+", str(any_chunk_of_text).lower())
stopwords = ["i", "me", "my", "myself", "we", "our", "ours", "ourselves", "you", "your", "yours", "yourself", "yourselves", "he", "him", "his", "himself", "she", "her", "hers", "herself", "it", "its", "itself", "they", "them", "their", "theirs", "themselves", "what", "which", "who", "whom", "this", "that", "these", "those", "am", "is", "are", "was", "were", "be", "been", "being", "have", "has", "had", "having", "do", "does", "did", "doing", "a", "an", "the", "and", "but", "if", "or", "because", "as", "until", "while", "of", "at", "by", "for", "with", "about", "against", "between", "into", "through", "during", "before", "after", "above", "below", "to", "from", "up", "down", "in", "out", "on", "off", "over", "under", "again", "further", "then", "once", "here", "there", "when", "where", "why", "how", "all", "any", "both", "each", "few", "more", "most", "other", "some", "such", "no", "nor", "not", "only", "own", "same", "so", "than", "too", "very", "s", "t", "can", "will", "just", "don", "should", "now", "ve", "ll", "amp"]

def load_course_comments():
    """Load the course YouTube-comments corpus into a DataFrame, one comment per row."""
    filename = "tc_youtube_comments.txt"
    text = None
    for path in (filename, os.path.join("notebooks", "data", filename),
                 os.path.join("data", filename), os.path.join("..", "data", filename)):
        if os.path.exists(path):
            text = open(path, encoding="utf-8").read(); break
    if text is None:
        url = "https://raw.githubusercontent.com/TCU-DCDA/WRIT20833_2026/main/notebooks/data/" + filename
        text = urllib.request.urlopen(url).read().decode("utf-8")
    rows = [line.strip() for line in text.split("\n") if line.strip()]
    return pd.DataFrame({"comment": rows})

# ----- Choose your data -----------------------------------------------------------
# EXPECTED PATH -- bring your own: the cleaned CSV from the Day 10 workshop (ideally the
# SAME dataset you used in HW3, so frequency + sentiment + topics all describe one corpus).
#     df = pd.read_csv("YOUR_FILE.csv")
#     text_column = "the_text_column"
#
# FALLBACK (runs out of the box) -- the course corpus on the TX Ten Commandments law:
df = load_course_comments()
text_column = "comment"
# ---------------------------------------------------------------------------------

# Sentiment carried over from HW3 (you already know this step):
df["sentiment"] = df[text_column].apply(lambda t: analyzer.polarity_scores(t)["compound"])

print(f"{len(df)} rows loaded and sentiment-scored. Text column: '{text_column}'.")
df.head()

## Part A — From comments to topics (5 exercises)

**A1 — Turn each comment into a list of meaningful words.**

In [ ]:
# A1 — solution
df["tokens"] = df[text_column].apply(
    lambda t: [w for w in split_into_words(t) if w and w not in stopwords])
print(df["tokens"].iloc[0])
# e.g. ['putting', 'constitution', 'classrooms']  -- HW2's "meaningful words", kept PER comment.

> **Teaching note:** Goal: HW2's filter, but the list is kept per-document (LDA needs documents,
> not one pile). Good moment to connect back: same `split_into_words` + `stopwords`, new shape.

**A2 — Build the dictionary and corpus.**

In [ ]:
# A2 — solution
dictionary = corpora.Dictionary(df["tokens"])
corpus = [dictionary.doc2bow(toks) for toks in df["tokens"]]
print("vocabulary size:", len(dictionary))   # ~650 unique words on the course corpus
print(corpus[0])                               # [(id, count), ...]
print([(dictionary[i], c) for i, c in corpus[0]])   # human-readable

# (Sidebar) the most negative comment from HW3 is long -> its bag has many words; a short
# comment's bag is tiny. Document length varies a lot in this corpus.

> **Teaching note:** Goal: the bag-of-words representation; word ORDER is discarded. The
> "dog bites man" example lands the loss of syntax/context. Vocabulary size depends on the corpus
> (~650 here) — not a fixed number to evaluate against.

**A3 — An edge case: predict, then run.**

In [ ]:
# A3 — solution
fragment_tokens = [w for w in split_into_words("The") if w and w not in stopwords]
print("tokens:", fragment_tokens)              # []  -- "the" is a stopword, so nothing remains
print("bag of words:", dictionary.doc2bow(fragment_tokens))   # []  -- an EMPTY document

> **Teaching note:** Goal: predict-then-run (the house motif), here showing an **empty bag of
> words**. Ties directly to HW3's data-quality finding: the corpus has wrap-fragments ("Of Texas",
> "The") that reduce to nothing. Empty docs get a meaningless/near-uniform topic — students handle
> them with the `-1` guard in B1. Real data is messy; the pipeline must not crash on it.

**A4 — Train the topic model.**

In [ ]:
# A4 — solution
NUM_TOPICS = 4
lda = LdaModel(corpus=corpus, id2word=dictionary, num_topics=NUM_TOPICS,
               random_state=42, passes=10)
for topic_id, words in lda.print_topics(num_words=6):
    print(f"topic {topic_id}: {words}")
# REPRESENTATIVE ONLY (yours will differ): on this single-issue corpus every topic is dominated
# by commandments / religion / god / ten / school -- the topics OVERLAP because the whole corpus
# is about one law. That overlap is itself the finding (see note).

> **Teaching note:** Goal: train + read LDA topics. Expect heavy overlap here — a narrow,
> single-issue corpus doesn't separate into crisp topics, which is an honest lesson about LDA's
> limits on small/homogeneous data (and an argument for students bringing a richer dataset). Don't
> evaluate by specific words; evaluate the student's *reading* of whatever they get.

**A5 — Read and *name* the topics (model answer).**

> **Model answer:** Any defensible labels with reasoning, e.g. "Topic A = church/state
> separation argument (church, state, religion); Topic B = think-of-the-children framing (children,
> schools, want); Topic C = scripture/law literalism (ten, commandments, law, bible)." Full credit
> for naming + flagging at least one **vague or mixed** topic. The point is that humans author the
> meaning; two students may label the same topics differently and both be right.

### A note: a topic is a *pattern*, not a fact — and you didn't build the math
Like VADER in HW3, **LDA is a borrowed tool** — you `pip install` gensim; nobody in this course
writes the statistics from scratch. But LDA differs from almost everything else you've run: it is
**stochastic**. Run it twice and the topics can come out in a different order, or shift, because it
starts from randomness (we set `random_state` to pin it down, but a different computer or gensim
version can still differ).

So a "topic" is **not a fact the data contains** — it's a *pattern the model proposes*, and a
**human decides** whether it means anything and what to call it (that's A5). Two analysts can label
the same topics differently. Hold your topics loosely. *(Day 7 again: judging a tool's output, not
just running it.)*

## Part B — Integration: topics meet sentiment (4 exercises)

**B1 — Assign each comment its dominant topic.**

In [ ]:
# B1 — solution
def dominant_topic(bow):
    topic_probs = lda.get_document_topics(bow)
    if not topic_probs:
        return -1
    return max(topic_probs, key=lambda pair: pair[1])[0]

df["topic"] = [dominant_topic(b) for b in corpus]
print(df[[text_column, "topic"]].head())

> **Teaching note:** Goal: dominant-topic assignment + the `-1` guard for empty docs (the A3
> case). Critical hook: forcing a multi-theme comment into ONE bin loses nuance — the same flattening
> theme as HW2/HW3, now at the topic level.

**B2 — How many comments per topic?**

In [ ]:
# B2 — solution
print(df["topic"].value_counts().sort_index())
df["topic"].value_counts().sort_index().plot(kind="bar", title="Comments per topic")
plt.xlabel("topic"); plt.ylabel("comments"); plt.show()
# Roughly 25-35 comments per topic on the course corpus (varies by run/version); a small "-1"
# group may appear for empty/fragment docs.

> **Teaching note:** Goal: one-line pandas viz again (reinforcement). Read distribution as
> evidence; flag the `-1` bar as the data-quality residue, not a real theme.

**B3 — The integration: how does each topic feel?**

In [ ]:
# B3 — solution
print(df.groupby("topic")["sentiment"].mean().round(3))
# REPRESENTATIVE ONLY (yours will differ): the per-topic means SPREAD OUT -- some topics run
# WARMER and some run COOLER (e.g. roughly 0.01 to 0.14 on one run; sign and order are not
# reproducible). The POINT is that different sub-conversations carry different average feeling.

> **Teaching note:** Goal: **the synthesis** — `groupby(topic)["sentiment"].mean()` joins HW4
> topics to HW3 sentiment. This is the capstone move in one line. The robust lesson (independent of
> the stochastic numbers): sentiment is *not uniform across topics* — some sub-conversations are
> angrier than others. Push students to check a topic's mean against its A4 words for coherence.

**B4 — Interpretation (model answer).**

> **Model answer (accept variants; require all three lenses + one named failure):** e.g. "The
> conversation is really several arguments under one law. The church/state topic (label, A5) runs
> cooler (lower B3 mean) than the 'teach values' topic — so *commandments*, the word that 'won'
> HW2's frequency count, is actually shared across a warmer camp and a cooler one.
> **Where the tools mislead:** Topic C looked vague (overlapping words), one long sarcastic comment
> is mis-scored by VADER, and the `-1` fragments are noise — I'd re-read those by hand before
> trusting the averages."

> **Teaching note:** This is the assignment's capstone-in-miniature. Reward integration of all three
> methods + an honest limitation, not a "correct" reading. Bridges straight into the capstone.

## Part C — Going deeper / capstone bridge (2 exercises)

**C1 — Change the number of topics (model).**

In [ ]:
# C1 — solution
for k in (2, 6):
    m = LdaModel(corpus=corpus, id2word=dictionary, num_topics=k, random_state=42, passes=10)
    print(f"\n=== {k} topics ===")
    for tid, words in m.print_topics(num_words=5):
        print(f"topic {tid}: {words}")
# With 2 topics the split is coarse (roughly pro/skeptical framings blur together); with 6 it
# fragments into near-duplicates. No "true" number -- the analyst authors the choice.

> **Teaching note:** Goal: show that `num_topics` is an authored knob, not a discovered truth
> (callback to HW1/HW3 threshold choices). Reward a student who can *defend* a chosen number for
> their own data over one who finds the "right" one.

**C2 — Capstone bridge (model).**

> **Model answer:** Any concrete plan that names (a) a dataset — ideally the student's own,
> richer than 123 short comments — (b) which lens fits their question (frequency for "what words
> dominate," sentiment for "how do they feel," topics for "what sub-conversations exist"), and (c) one
> honest limitation. No code required.

> **Teaching note:** This *is* the capstone proposal in embryo. For BYO submissions, evaluate the fit
> between question and method, and the candor about limits — not the dataset's size or "results."